In [1]:
!pip install --disable-pip-version-check -q optuna

In [25]:
import json
import sys
import time
from datetime import datetime

import boto3
import numpy as np
import pandas as pd
import sagemaker
from sagemaker.model_metrics import ModelMetrics
from sklearn.metrics import fbeta_score, accuracy_score, precision_score, recall_score, f1_score

sys.path.append('../config')
import config

In [8]:
# Set up AWS session
session = boto3.session.Session()
region = session.region_name
sagemaker_session = sagemaker.Session()
s3_client = boto3.client('s3')
sm_client = boto3.client('sagemaker')
role = sagemaker.get_execution_role()

In [19]:
# Set up S3 vars
bucket = config.S3_BUCKET
s3_prefix = 'final_project/feature_engineer'
model_prefix = 'final_project/models'
output_prefix = 'final_project/output'

train_path = f's3://{bucket}/{s3_prefix}/Xy_train.csv'
val_path = f's3://{bucket}/{s3_prefix}/Xy_val.csv'

In [4]:
# Load data from S3
def load_data_from_s3(s3_path):
    bucket_name = bucket
    key = s3_path.replace(f's3://{bucket_name}/', '')
    obj = s3_client.get_object(Bucket=bucket_name, Key=key)
    return pd.read_csv(obj['Body'])


print(f"Attempting to load training data from {train_path}.")
train_df = load_data_from_s3(train_path)
print(f"Attempting to load validation data from {val_path}.")
val_df = load_data_from_s3(val_path)

print(train_df.head())
train_df = train_df.replace([np.inf, -np.inf], np.nan)
val_df = val_df.replace([np.inf, -np.inf], np.nan)
train_df.dropna(inplace=True)
val_df.dropna(inplace=True)

print(f"Training data shape: {train_df.shape}")
print(f"Validation data shape: {val_df.shape}")

X_train = train_df.drop(['label', 'record_id'], axis=1)
y_train = train_df['label']
X_val = val_df.drop(['label', 'record_id'], axis=1)
y_val = val_df['label']

val_features_local_path = '../data/tmp/validation_features.csv'
X_val.to_csv(val_features_local_path, header=False, index=False)
val_features_s3_key = f'{s3_prefix}/validation_features.csv'
s3_client.upload_file(val_features_local_path, bucket, val_features_s3_key)
val_features_path = f's3://{bucket}/{val_features_s3_key}'

Attempting to load training data from s3://sagemaker-us-east-1-637423636147/final_project/feature_engineer/Xy_train.csv.
Attempting to load validation data from s3://sagemaker-us-east-1-637423636147/final_project/feature_engineer/Xy_val.csv.
   subflow_fwd_bytes  avg_fwd_segment_size  fwd_packet_length_max  \
0                 60             30.000000                     30   
1              12982           1854.571400                   7215   
2                 24              6.000000                      6   
3                 26              8.666667                     20   
4                 56              7.000000                     20   

   fwd_act_data_packets  fwd_packet_length_mean   fwd_iat_std  \
0                     1               30.000000  0.000000e+00   
1                     5             1854.571400  3.200000e+07   
2                     3                6.000000  5.420316e+06   
3                     2                8.666667  4.101220e+02   
4                 

In [5]:
def save_data_for_sagemaker(X, y, filename):
    data = pd.concat([y, X], axis=1)
    local_path = f'../data/tmp/{filename}'
    data.to_csv(local_path, header=False, index=False)
    s3_client.head_object(Bucket=bucket, Key=f'{s3_prefix}/Xy_train.csv')
    s3_key = f'{s3_prefix}/{filename}'
    s3_client.upload_file(local_path, bucket, s3_key)
    return f's3://{bucket}/{s3_key}'


train_path = save_data_for_sagemaker(X_train, y_train, 'train.csv')
val_path = save_data_for_sagemaker(X_val, y_val, 'validation.csv')

print(f"Training data saved to: {train_path}")
print(f"Validation data saved to: {val_path}")

Training data saved to: s3://sagemaker-us-east-1-637423636147/final_project/feature_engineer/train.csv
Validation data saved to: s3://sagemaker-us-east-1-637423636147/final_project/feature_engineer/validation.csv


In [ ]:
# Train model
job_name = f"xgboost-tuning-{int(time.time())}"
hyperparameters = {
    'num_round': 100,
    'objective': 'binary:logistic',
}
xgb = sagemaker.estimator.Estimator(
    image_uri=sagemaker.image_uris.retrieve("xgboost", region, "1.5-1"),
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    volume_size=50,
    output_path=f's3://{bucket}/{output_prefix}',
    sagemaker_session=sagemaker_session,
    hyperparameters=hyperparameters
)

train_data = sagemaker.inputs.TrainingInput(
    train_path,
    distribution="FullyReplicated",
    content_type="text/csv",
    s3_data_type="S3Prefix",
)

validation_data = sagemaker.inputs.TrainingInput(
    val_path,
    distribution="FullyReplicated",
    content_type="text/csv",
    s3_data_type="S3Prefix",
)

xgb.fit(
    inputs={'train': train_data, 'validation': validation_data},
    job_name=job_name,
    wait=True
)

In [20]:
# Create model and transformer for validation batch prediction
model = sagemaker.model.Model(
    image_uri=xgb.image_uri,
    model_data=xgb.model_data,
    role=role,
    sagemaker_session=sagemaker_session
)

transformer = model.transformer(
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{bucket}/{output_prefix}/predictions',
    accept='text/csv'
)

transformer.transform(
    val_features_path,
    content_type='text/csv',
    split_type='Line',
)
transformer.wait()

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2025-06-08-13-13-55-280
INFO:sagemaker:Creating transform job with name: sagemaker-xgboost-2025-06-08-13-13-56-359


..............................
/miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2025-06-08:13:19:05:INFO] No GPUs detected (normal if no gpus installed)
[2025-06-08:13:19:05:INFO] No GPUs detected (normal if no gpus installed)
[2025-06-08:13:19:05:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
/miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2025-06-08:13:19:05:INFO] No GPUs detected (normal if no gpus installed)
[2025-06-08:13:19:05:INFO] No GPUs detected

In [21]:
transform_job_name = transformer.latest_transform_job.name
transform_job_desc = sm_client.describe_transform_job(TransformJobName=transform_job_name)
job_status = transform_job_desc['TransformJobStatus']

print(f"Transform job {transform_job_name} status: {job_status}")

if job_status != 'Completed':
    print(f"Transform job failed with status: {job_status}")
    if 'FailureReason' in transform_job_desc:
        print(f"Failure reason: {transform_job_desc['FailureReason']}")
    exit(1)

Transform job sagemaker-xgboost-2025-06-08-13-13-56-359 status: Completed


In [26]:
# Perform basic evaluation on validation dataset
def load_predictions_from_s3():
    prediction_key = f"{output_prefix}/predictions/validation_features.csv.out"
    print(f"Loading predictions from S3: {prediction_key}")
    obj = s3_client.get_object(Bucket=bucket, Key=prediction_key)
    predictions_df = pd.read_csv(obj['Body'], header=None)
    return predictions_df[0].values

predictions_prob = load_predictions_from_s3()
predictions_binary = (predictions_prob > 0.5).astype(int)
true_labels = y_val.values

accuracy = accuracy_score(true_labels, predictions_binary)
precision = precision_score(true_labels, predictions_binary)
recall = recall_score(true_labels, predictions_binary)
f1 = f1_score(true_labels, predictions_binary)
f2 = fbeta_score(true_labels, predictions_binary, beta=2)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"F2 Score:  {f2:.4f}")

evaluation_metrics = {
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'f2_score': float(f2)
}


Loading predictions from S3: final_project/output/predictions/validation_features.csv.out
Accuracy:  0.9993
Precision: 0.9998
Recall:    0.9990
F1 Score:  0.9994
F2 Score:  0.9991


In [27]:
# Store model info
model_info = {
    'model_data': xgb.model_data,
    'training_job_name': xgb.latest_training_job.name,
    'hyperparameters': hyperparameters,
    'evaluation_metrics': evaluation_metrics,
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

model_info_key = f'{model_prefix}/model_info.json'
s3_client.put_object(
    Body=json.dumps(model_info, indent=2),
    Bucket=bucket,
    Key=model_info_key
)

{'ResponseMetadata': {'RequestId': 'KTGZ6YRMMTAC1PDH',
  'HostId': 'dEH9GcfbuLPdlKIEUDkZ0j8cS/9O8TE8UonEyDwOYibtNMWswvvvnvXqPb6/FPuztLsrUfuSmBQ=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'dEH9GcfbuLPdlKIEUDkZ0j8cS/9O8TE8UonEyDwOYibtNMWswvvvnvXqPb6/FPuztLsrUfuSmBQ=',
   'x-amz-request-id': 'KTGZ6YRMMTAC1PDH',
   'date': 'Sun, 08 Jun 2025 13:21:42 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"b5584a451bb9495a80a1f636cd64e88f"',
   'x-amz-checksum-crc32': 'qraDkw==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"b5584a451bb9495a80a1f636cd64e88f"',
 'ChecksumCRC32': 'qraDkw==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

In [ ]:
# Create model package group
model_package_group_name = f"ddos-traffic-prediction-model-group-{int(time.time())}"
try:
    sm_client.create_model_package_group(
        ModelPackageGroupName=model_package_group_name,
        ModelPackageGroupDescription="DDoS Traffic Prediction Models"
    )
except sm_client.exceptions.ResourceInUse:
    print(f"Model package group {model_package_group_name} already exists")

model_package_arn = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name=model_package_group_name,
    approval_status="Approved",
    model_metrics=ModelMetrics(
        model_statistics=None,
        model_data_statistics=None,
        bias=None,
        explainability=None,
    )
)